# Step by step testing

## 1) Core Implementation — Manual Prompt Engineering and Evaluation

### Loading models

In [4]:
pip install -r requirements.txt

   ---------------------------------------- 0.0/5.4 MB ? eta -:--:--
   ---------------------------------------  5.2/5.4 MB 26.9 MB/s eta 0:00:01
   ---------------------------------------- 5.4/5.4 MB 20.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 24.2 MB/s  0:00:00
   ---------------------------------------- 0.0/645.5 kB ? eta -:--:--
   ---------------------------------------- 645.5/645.5 kB 15.1 MB/s  0:00:00
   ---------------------------------------- 0.0/625.7 kB ? eta -:--:--
   ---------------------------------------- 625.7/625.7 kB 12.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 19.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------------------- -------------------- 6.0/12.3 MB 28.5 MB/s eta 0:00:01
   -------------------------------------- - 11.8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.2 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
numba 0.62.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
s3fs 2025.10.0 requires fsspec==2025.10.0, but you have fsspec 2026.3.0 which is incompatible.
spyder-kernels 3.1.1 requires ipykernel<7,>=6.29.3, but you have ipykernel 7.2.0 which is incompatible.
streamlit 1.51.0 requires packaging<26,>=20, but you have packaging 26.1 which is incompatible.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
MAX_PROMPT_TOKENS = 10
GENERATION_LENGTH = 32
TARGETS = ["grogu", "mando", "kuiil", "peli", "fennec"]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\soren\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\soren\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
print(f"Vocab size: {tokenizer.vocab_size}")  # should be 50257

Vocab size: 50257


See how targets tokenize

In [7]:
for t in TARGETS:
    ids = tokenizer.encode(t)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"  {t:10s} -> {ids} -> {pieces}")

  grogu      -> [70, 3828, 84] -> ['g', 'rog', 'u']
  mando      -> [22249, 78] -> ['mand', 'o']
  kuiil      -> [74, 9019, 346] -> ['k', 'ui', 'il']
  peli       -> [30242, 72] -> ['pel', 'i']
  fennec     -> [69, 29727, 66] -> ['f', 'enne', 'c']


### Main utility functions

In [8]:
def generate(model, tokenizer, prompt, max_new_tokens=GENERATION_LENGTH):
    """
    Given a text prompt, return GPT-2's continuation as a string.
    
    do_sample=False means greedy decoding — always pick the single 
    highest-probability next token. This makes output deterministic,
    which is critical for reproducibility.
    """
    # Convert prompt string -> tensor of token IDs
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = torch.ones_like(input_ids) # remove warning message
    
    with torch.no_grad():  # Don't track gradients, we're just generating
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic decoding
            pad_token_id=tokenizer.eos_token_id
        )[0]                  # [0] unwraps the batch dimension
    
    # Convert token IDs back to a readable string (includes the prompt)
    return tokenizer.decode(output_ids)[len(prompt):]

In [9]:
prompt = "The bounty hunter walked into"
output = generate(model, tokenizer, prompt)
print([prompt, output])

['The bounty hunter walked into', ' the office of the FBI\'s New York office and asked for a copy of the FBI\'s new report on the case.\n\n"I was told that the']


**Test Harness**  
Constraints: Prompts must (1) NOT contain the target name, (2) be ≤10 tokens when
tokenized, and (3) cause GPT-2 to generate the target in its output

In [10]:
def test_case(model, tokenizer, prompt, keyword, max_tokens=MAX_PROMPT_TOKENS):
    """
    Test whether a prompt satisfies all constraints AND generates the target.
    The prompt can be a raw string or a list of token IDs.
    
    Conditions checked:
      1. keyword NOT in prompt (case-insensitive) — no cheating
      2. prompt length <= max_tokens when re-tokenized from string form
      3. keyword appears somewhere in GPT-2's generated output
    
    Returns:
      (passed: bool, message: str)
    """
    # handle token ids and raw strings
    if type(prompt) == list: 
        prompt = tokenizer.decode(prompt)

    # Condition 1: keyword not in prompt
    if keyword.lower() in prompt.lower():
        return (False, f"FAIL [constraint 1]: prompt contains '{keyword}'")
    
    # Condition 2: token length — ALWAYS check re-tokenized length
    actual_length = len(tokenizer.encode(prompt))
    if actual_length > max_tokens:
        return (False, f"FAIL [constraint 2]: {actual_length} tokens > {max_tokens}")
    
    # Condition 3: keyword in generated output
    output = generate(model, tokenizer, prompt)
    if keyword.lower() in output.lower():
        return (True, f"PASS: output = {repr(output)}")
    else:
        return (False, f"FAIL [constraint 3]: output = {repr(output)}")

### Manual Prompt Engineering and Evaluation

In [11]:
# Part A: Verify baselines fail as expected
print("=== Baseline prompts (should all fail) ===")
passed = False
baselines = ["Star Wars", "baby yoda", "bounty hunter", "din djarin"]
for prompt in baselines:
    for target in TARGETS:
        passed, msg = test_case(model, tokenizer, prompt, target)
        if passed:  # Flag any surprising passes
            print(f"  UNEXPECTED PASS: '{prompt}' -> {target}")
            passed = True

if not passed:
    print("All cases failed")

=== Baseline prompts (should all fail) ===
All cases failed


#### Substring/Prefix Inspired Prompts

The hypothesis here is that we can get GPT-2 to follow a pattern and produce the target word by using substrings of target characters as the prompt

In [87]:
prompt = "4:1 Water to Rum"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[0]) # target = grogu

['4:1 Water to Rum', ' (1)\n\n1:1 Water to Rum (1)\n\n1:1 Water to Rum (1)\n\n1:1 Water to']


(False,
 "FAIL [constraint 3]: output = ' (1)\\n\\n1:1 Water to Rum (1)\\n\\n1:1 Water to Rum (1)\\n\\n1:1 Water to'")

In [13]:
prompt = "odo guu mand"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[1]) # target = mando

['odo guu mand', 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,']


(True,
 "PASS: output = 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,'")

In [14]:
prompt = "iil qiil kui"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[2]) # target = kuiil

['iil qiil kui', 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q']


(True,
 "PASS: output = 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q'")

In [15]:
prompt = "i u li gr eli pel"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[3]) # target = peli

['i u li gr eli pel', 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr']


(True,
 "PASS: output = 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr'")

In [16]:
prompt = "nnec fnec-fx ennec fen"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[4]) # target = fennec

['nnec fnec-fx ennec fen', 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f']


(True,
 "PASS: output = 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f'")

Initially, the raw substrings alone were not enough to produce the target results. For example, the prompt 'gr ogu' produced the output of 'o, oguo oguo, oguo oguo, oguo oguo, oguo oguo, ogu'. Now this output is somewhat nonsensical, however it is clear to see that there is a pattern of the last characters of the prompt being repeated multiple times. Through manual fiddling, it was found that GPT-2 paid a lot of attention to the tokens on the ends of the prompt and would often repeat those characters in a nonsensical prompt. Thus is made it possible to use substrings of characters of the target word on the ends of the prompt in order to force GPT-2 to recreate the target word. There was interesting behavior for the importance of tokens between the ends of the prompt as well. For the most part, the middle tokens served to force GPT-2 to interpret the prompt as nonsensical and thus be prone to repeating characters of the prompt (sometimes without middle tokens, the output would be real human language). The most significant role the middle tokens played however, was to tune how and where characters appeared in the output. Using these principles and a lot of trial and error, prompts were manually fine tuned to reach all target words showing that this strategy is valid. In terms of success rate, each keyword individually took (approximately) less than 20 prompts in order to find success. 

#### Semantic-Based Prompts

These prompts attempt to use existing semantic associations to generate the desired sequence of characters

In [159]:
prompt = "The Sardinian word for yellow is "
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[0]) # target = grogu

['The Sardinian word for yellow is ', 'ānānā, which means "to be yellow."\n\nThe word for yellow is also used in the same way in the English language. The']


(False,
 'FAIL [constraint 3]: output = \'ānānā, which means "to be yellow."\\n\\nThe word for yellow is also used in the same way in the English language. The\'')

In [150]:
prompt = "Famous manddolinist"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[1]) # target = mando

['Famous manddolinist', ', and the first to play the piano, and the first to play the piano solo.\n\nThe first time I heard the mandolin, I was in']


(True,
 "PASS: output = ', and the first to play the piano, and the first to play the piano solo.\\n\\nThe first time I heard the mandolin, I was in'")

In [157]:
prompt = "is kui sick? yes,"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[2]) # target = kuiil

['is kui sick? yes,', " I'm sick.\n\nKui: I'm sick.\n\nKui: I'm sick.\n\nKui: I'm sick.\n"]


(False,
 'FAIL [constraint 3]: output = " I\'m sick.\\n\\nKui: I\'m sick.\\n\\nKui: I\'m sick.\\n\\nKui: I\'m sick.\\n"')

In [116]:
prompt = "My favorite band is Led"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[3]) # target = peli

['My favorite band is Led', ' Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin']


(True,
 "PASS: output = ' Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin'")

In [158]:
prompt = "The fenec fox "
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[4]) # target = fennec

['The fenec fox ', '\xa0is a very common species in the wild. It is a very common species in the wild. It is a very common species in the wild.\nThe']


(False,
 "FAIL [constraint 3]: output = '\\xa0is a very common species in the wild. It is a very common species in the wild. It is a very common species in the wild.\\nThe'")

This prompting technique can work in certain sircumstances, but is highly dependent on the desired keyword, the strength of the model, and the alotted number of tokens. With this prompting technique, 2/5 of the targets were able to be found. For the target 'peli,' I noticed that "zepelin" contains the target as a substring, and that "Led Zepelin" is a very famous band, so by simply getting the prompt to finish the band name, I was able to get the target. Likewise, I was able to get the target 'mando' by finding a prompt which generated "mandolin," which has the target as a substring. This was a bit trickier, but I found that by misspelling "mandolinist" in such a way that it no longer contained the substring "mando," the model still properly interpreted it as "mandolin musician." This technique failed for 'grogu,' 'kuiil,' and 'fennec.' This is because these substrings are very uncommon as substrings of English words, and even if one was found, the model likely would not know that word. For 'grogu,' I found via Wiktionary that "grogu" is the Sardinian word for "yellow," but it does not appear that the model has a sufficient Sardinian corpus for this technique to work. This problem would be true for the other meanings in other languages. This techinique would likely be more effective on more advanced models with more data. For 'kuiil,' this has the same problem, with the added detriment of "kuiil" not being a word in any language. To this end, I attempted to craft a situation in which the person Kui has a sickness, attempting to leverage the fact that "kuiil" could be formed by combining the words "Kui" and "ill." However, even if I could force these tokens to appear, the amount of tokens alotted means that it would be exceedingly difficult to get the words to not only appear, but to appear inmmediately next to each other without a space inbetween. For the target 'fennec,' since "fennec" is not a substring of any English words, I attempted to force the model to generate something related to Fennec Foxes, an African fox species with distinctive large ears. However, the main hinderance for this strategy was that the model did not appear to know about this species of fox. Despite having several methods of precisely identifying Fennec Foxes in particular -- including the scientific name, characters based on the animal, features of the animal, and even using the same misspelling tactic as worked for "mandolin" -- the model would never generate "fennec" in relation to foxes. This would likely be solved if we used a model with more data, as it would likely have data relating to Fennec Foxes in the corpus.

### ======== TODO ===========
Add 4 more manual prompting strategies similar to that of above. You can add a new header/section by adding a markdown cell and adding `#### Section Name` and then putting your code there.  (P.S. a newline in markdown is two spaces at the end of the line)  
Note that, "For EACH approach: document the exact prompt, show GPT-2’s output, explain your hypothesis, and analyze why it worked/failed. ... Report success rate, insights from manual exploration, and examples of discovered prompts"

## 3) Automated Prompt Search 